In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/data_processed_with_elo.csv")
df["date"] = pd.to_datetime(df["date"])
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,elo_home,elo_away,elo_diff
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1500.000000,1500.000000,0.000000
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1494.000000,1506.000000,-12.000000
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1499.792850,1500.207150,-0.414301
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1500.199996,1499.800004,0.399991


In [3]:
home = df[["date", "home_team", "elo_home"]].rename(
    columns={"home_team": "team", "elo_home": "elo"}
)

away = df[["date", "away_team", "elo_away"]].rename(
    columns={"away_team": "team", "elo_away": "elo"}
)

all_elo = pd.concat([home, away])

In [28]:
elo_dict = (
    all_elo.sort_values("date")
    .groupby("team")
    .tail(1)
    .set_index("team")["elo"]
    .to_dict()
)

In [7]:
df = df.sort_values("date")

In [8]:
# Target
# 0 = away win
# 1 = draw
# 2 = home win
def get_results(row):
    if row["home_score"] > row["away_score"]:
        return 2
    elif row["home_score"] < row["away_score"]:
        return 0
    else:
        return 1

df["target"] = df.apply(get_results, axis=1)

In [9]:
# Encoding
df = pd.get_dummies(df, columns=["tournament"], drop_first=True)

In [10]:
df = df.drop(columns=[
    "home_team",
    "away_team",
    "city",
    "country"
])

In [11]:
train = df[df["date"] < "2018-01-01"]
test = df[df["date"] >= "2018-01-01"]

In [13]:
# Features para ML
"""feature_cols = [
    "elo_home",
    "elo_away",
    "elo_diff",
    "neutral"
] + [c for c in df.columns if "tournament_" in c]"""

feature_cols = [
    "elo_diff",
    "neutral"
] + [c for c in df.columns if "tournament_" in c]

X_train = train[feature_cols]
X_test = test[feature_cols]

y_train = train["target"]
y_test = test["target"]

In [14]:
# Escalar datos
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
# Alinear columnas
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [16]:
# Evaluar modelo
from sklearn.metrics import accuracy_score
pred = model.predict(X_test_scaled)
accuracy_score(y_test, pred)

0.594423731561031

In [17]:
# Comparar Baselina Elo
elo_pred = (test["elo_home"] > test["elo_away"]).astype(int)

In [18]:
model.predict(X_test_scaled)

array([0, 2, 0, ..., 0, 2, 2], shape=(6169,))

In [19]:
elo_pred = (test["elo_home"] > test["elo_away"]).astype(int)

In [20]:
ml_pred = model.predict(X_test_scaled)

In [21]:
from sklearn.metrics import accuracy_score
print("Elo baseline:", accuracy_score(y_test, elo_pred))
print("ML model:", accuracy_score(y_test, ml_pred))

Elo baseline: 0.3400875344464257
ML model: 0.594423731561031


In [22]:
from sklearn.metrics import log_loss
probs = model.predict_proba(X_test_scaled)
log_loss(y_test, probs)

0.8889712728803764

In [62]:
# Guardar Modelo
import joblib
joblib.dump(model, "../models/model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(features_cols, "../models/features.pkl")
joblib.dump(elo_dict, "../models/elo_dict.pkl")

['../models/elo_dict.pkl']

In [59]:
# Funcio para predecir partidos
def predic_match(team_a, team_b, elo_dict, scaler, model, neutral=1, tournament="FIFA World Cup"):
    import pandas as pd

    elo_a = elo_dict[team_a]
    elo_b = elo_dict[team_b]
    elo_diff = elo_a - elo_b
    X = pd.DataFrame(0, columns=feature_cols, index=[0])
    X["elo_diff"] = elo_diff
    X["neutral"] = neutral
    tournament_col = f"tournament_{tournament}"
    if tournament_col in X.columns:
        X[tournament_col] = 1

    X_scaled = scaler.transform(X)

    probs = model.predict_proba(X_scaled)[0]

    team_a_win = probs[2]
    draw = probs[1]
    team_b_win = probs[0]

    print(f"\n{team_a} vs {team_b}\n")
    print(f"{team_a} gana: {team_a_win*100:.2f}%")
    print(f"Empate: {draw*100:.2f}%")
    print(f"{team_b} gana: {team_b_win*100:.2f}%")

    return {
        "team_a": team_a,
        "team_b": team_b,
        "tournament": tournament,
        "team_a_win": float(team_a_win),
        "draw": float(draw),
        "team_b_win": float(team_b_win)
    }

In [61]:
# prueba
predic_match(
    "Argentina",
    "Algeria",
    elo_dict,
    scaler,
    model
)


Argentina vs Algeria

Argentina gana: 81.14%
Empate: 13.17%
Algeria gana: 5.69%


{'team_a': 'Argentina',
 'team_b': 'Algeria',
 'tournament': 'FIFA World Cup',
 'team_a_win': 0.8113924152912408,
 'draw': 0.1317190218715404,
 'team_b_win': 0.056888562837218805}